# Notebook 10: Real-World Agents with APIs

**🎯 Goal:** Learn how to connect agents to the real world by giving them tools that interact with external services and APIs like web search and Wikipedia.

## 🧩 Concept Overview

An agent's intelligence is limited by the information and actions available to it. So far, our tools have been simple Python functions. To make agents truly powerful, we need to connect them to the vast resources of the internet and other external services.

This is done by creating tools that act as wrappers around public or private APIs. This allows an agent to:
- **Search the web** for up-to-date information.
- **Query a database** for specific data.
- **Look up information** in a knowledge base like Wikipedia.
- **Interact with services** like Slack, Gmail, or Notion.

## 🖼️ Visual Diagram

An agent using an external API tool:

```
                 +----------------+
Agent's Thought: | "I need to know | --> Action: Use WikipediaTool
                 | about Einstein"|           with query: "Albert Einstein"
                 +----------------+
                         ^
                         | (Final Answer)
                         |
                 +----------------+
Observation:     | "Albert Einstein| <-- Tool Call to Wikipedia API <-- [Wikipedia Server]
                 | was a physicist.."|
                 +----------------+
```

## ⚙️ Setup

We need to install the packages for the specific integrations we want to use. We'll use DuckDuckGo for web search (which doesn't require an API key) and Wikipedia.

In [ ]:
# pip install duckduckgo-search wikipedia

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

# Import the tools
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 1. Using Pre-built Tools

LangChain has a large collection of pre-built tools for many popular services. Using them is as simple as importing and initializing them.

In [ ]:
# 🌐 Web Search Tool
search_tool = DuckDuckGoSearchRun()

# 📚 Wikipedia Tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=2000)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

# Let's test them directly
search_result = search_tool.run("latest news on OpenAI")
print(f"--- Search Result ---\n{search_result}\n")

wiki_result = wiki_tool.run("LangChain")
print(f"--- Wikipedia Result ---\n{wiki_result}")

## 2. Building an Agent with Real-World Tools

Now let's give these tools to an agent and let it decide which one to use.

In [ ]:
tools = [search_tool, wiki_tool]

# Get the ReAct prompt template
prompt = hub.pull("hwchase17/react")

# Create the agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

query = "What is the latest news about the company that created LangChain, and what is LangChain according to Wikipedia?"
agent_executor.invoke({"input": query})

## 3. Connecting to a Database (Conceptual)

The same principles apply to other data sources, like databases or CSV files. LangChain has specialized agents that are highly optimized for these tasks.

- `create_pandas_dataframe_agent`: For asking questions about data in a pandas DataFrame.
- `create_sql_agent`: For converting natural language into SQL queries to run on a database.

Here's a conceptual example of how you might use the pandas agent with our `movie-plots.csv` file. Running this requires additional libraries like `tabulate`.

In [ ]:
import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent

# df = pd.read_csv('../data/movie-plots.csv')
# pandas_agent = create_pandas_dataframe_agent(llm, df, verbose=True)

# try:
#     # This agent can understand the dataframe and write python code to answer questions
#     pandas_agent.invoke({"input": "Which movie has the longest plot description?"})
# except ImportError as e:
#     print(f"Could not run pandas agent, likely missing dependencies like 'tabulate'. Error: {e}")
print("Pandas agent is a powerful tool for data analysis, allowing natural language queries on your dataframes.")

## 🔧 Hands-On Exercise

**Goal:** Build a dedicated Wikipedia research agent.

1.  Create a list containing only the `wiki_tool`.
2.  Create an `AgentExecutor` for this single-tool agent.
3.  Ask it two different questions:
    - `"Tell me about the Roman Empire."`
    - `"What is photosynthesis?"`

In [ ]:
# 1. Create the tool list
exercise_tools = [wiki_tool]

# 2. Create the agent executor
wiki_agent = create_react_agent(llm, exercise_tools, prompt)
wiki_executor = AgentExecutor(agent=wiki_agent, tools=exercise_tools, verbose=True)

# 3. Ask questions
print("--- Question 1: Roman Empire ---")
wiki_executor.invoke({"input": "Tell me about the Roman Empire."})

print("\n--- Question 2: Photosynthesis ---")
wiki_executor.invoke({"input": "What is photosynthesis?"})

## 🐞 Bug Bounty

A frequent issue when using external APIs is a missing or invalid API key. While DuckDuckGo and Wikipedia are free, many services (like Google Search via `SerpAPI`) are not.

**Task:** The code below shows how you would initialize a tool that requires an API key (`SerpAPI`). If you were to run this without setting the `SERPAPI_API_KEY` in your `.env` file, it would raise an `AuthenticationError`. The code is wrapped in a `try/except` block to prevent it from crashing the notebook.

In [ ]:
from langchain_community.utilities import SerpAPIWrapper

try:
    # This will fail if SERPAPI_API_KEY is not set
    search = SerpAPIWrapper()
    result = search.run("this will not run")
except Exception as e:
    print(f"Caught expected error: {e}")
    print("\nThis demonstrates the importance of setting API keys for protected services.")

## 💡 Pro Tip

Before building a custom tool to wrap a popular API, always check the official LangChain documentation and the `langchain-community` package. There's a high probability that someone has already built a robust, well-maintained integration for the service you need, saving you a significant amount of time and effort.

## 🏁 Summary

You've now unlocked the true potential of agents by connecting them to the outside world.

1.  **APIs are the Bridge:** Tools that call external APIs are what allow agents to access real-time, dynamic, and proprietary information.
2.  **LangChain Has a Rich Tool Ecosystem:** There are many pre-built tools for common services like web search, Wikipedia, and more.
3.  **Specialized Agents Exist for Specialized Tasks:** For common but complex tasks like data analysis, use specialized agents like `create_pandas_dataframe_agent` that are optimized for the job.